# CS-4063 — Natural Language Processing
## Assignment 3: Transformer-based Review Understanding with RAG-Enhanced Explanation Generation

## Imports and Setup

In [1]:
import os, gzip, json, math, random, re, time, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


Using device: cuda


## 1. Dataset Loading and Preprocessing

In [2]:
DATA_FILES = {
    'beauty':      'data/beauty.json.gz',
    'cellphones':  'data/cellphones.json.gz',
    'electronics': 'data/electronics.json.gz',
    'home':        'data/home.json.gz',
    'sports':      'data/sports.json.gz',
}
SELECTED_CATEGORIES = ['sports', 'cellphones', 'electronics']
REVIEWS_PER_CATEGORY = 12000   
MAX_SEQ_LEN = 128
MIN_FREQ    = 3                

def load_amazon_reviews(filepath, category, max_reviews=12000):
    """Load Amazon review JSON.gz file and extract text + rating."""
    reviews = []
    with gzip.open(filepath, 'rt', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if i >= max_reviews * 3:   # read extra to allow filtering
                break
            try:
                obj = json.loads(line.strip())
                text   = obj.get('reviewText', '') or obj.get('summary', '')
                rating = obj.get('overall', None)
                if text and rating and len(text.split()) >= 5:
                    reviews.append({
                        'text': text,
                        'rating': int(float(rating)),
                        'category': category
                    })
                    if len(reviews) >= max_reviews:
                        break
            except Exception:
                continue
    return reviews

print('Loading dataset...')
all_reviews = []
for cat in SELECTED_CATEGORIES:
    path = DATA_FILES[cat]
    if os.path.exists(path):
        reviews = load_amazon_reviews(path, cat, REVIEWS_PER_CATEGORY)
        print(f'  {cat}: loaded {len(reviews)} reviews')
        all_reviews.extend(reviews)
    else:
        print(f'  WARNING: {path} not found — skipping {cat}')

print(f'Total reviews loaded: {len(all_reviews)}')
df = pd.DataFrame(all_reviews)
print(df['rating'].value_counts().sort_index())

Loading dataset...
  sports: loaded 12000 reviews
  cellphones: loaded 12000 reviews
  electronics: loaded 12000 reviews
Total reviews loaded: 36000
rating
1     2102
2     1751
3     3212
4     7430
5    21505
Name: count, dtype: int64


In [3]:
def clean_text(text):
    """Text cleaning: lowercase, remove HTML, punctuation normalisation."""
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)           # strip HTML tags
    text = re.sub(r'http\S+|www\.\S+', ' ', text)  # strip URLs
    text = re.sub(r'[^a-z0-9\s\'\-]', ' ', text)  # keep letters/digits/apostrophe/hyphen
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    """Simple whitespace tokenizer with basic contraction splitting."""
    text = re.sub(r"(n't|'re|'ve|'ll|'d|'m|'s)", r' \1', text)
    return text.split()

# Apply cleaning
df['clean_text'] = df['text'].apply(clean_text)
df['tokens']     = df['clean_text'].apply(tokenize)

def rating_to_sentiment(r):
    if r <= 2: return 0
    if r == 3: return 1
    return 2
df['sentiment'] = df['rating'].apply(rating_to_sentiment)

def length_category(tokens):
    n = len(tokens)
    if n < 20: return 0
    if n <= 80: return 1
    return 2
df['length_cat'] = df['tokens'].apply(length_category)

print('Sentiment distribution:')
print(df['sentiment'].value_counts().sort_index())
print('\nLength category distribution:')
print(df['length_cat'].value_counts().sort_index())

Sentiment distribution:
sentiment
0     3853
1     3212
2    28935
Name: count, dtype: int64

Length category distribution:
length_cat
0      653
1    21869
2    13478
Name: count, dtype: int64


In [ ]:
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
n = len(df)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
train_df = df.iloc[:n_train].reset_index(drop=True)
val_df   = df.iloc[n_train:n_train+n_val].reset_index(drop=True)
test_df  = df.iloc[n_train+n_val:].reset_index(drop=True)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

counter = Counter(token for tokens in train_df['tokens'] for token in tokens)
SPECIAL = ['<PAD>', '<UNK>']
vocab_words = SPECIAL + [w for w, c in counter.most_common() if c >= MIN_FREQ]
word2idx = {w: i for i, w in enumerate(vocab_words)}
idx2word = {i: w for w, i in word2idx.items()}
VOCAB_SIZE = len(vocab_words)
PAD_IDX = word2idx['<PAD>']
UNK_IDX = word2idx['<UNK>']
print(f'Vocabulary size: {VOCAB_SIZE}')

def encode(tokens, max_len=MAX_SEQ_LEN):
    """Convert tokens to padded/truncated index sequence."""
    ids = [word2idx.get(t, UNK_IDX) for t in tokens[:max_len]]
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids

train_df['input_ids'] = train_df['tokens'].apply(encode)
val_df['input_ids']   = val_df['tokens'].apply(encode)
test_df['input_ids']  = test_df['tokens'].apply(encode)

with open('results/vocab.pkl', 'wb') as f:
    pickle.dump({'word2idx': word2idx, 'idx2word': idx2word}, f)
print('Vocabulary saved.')

Train: 25200 | Val: 5400 | Test: 5400
Vocabulary size: 16965
Vocabulary saved.


## Part A — Encoder Model for Understanding